In [ ]:
#Q3(a):Report measured bias squared and variance for the selected single tree, bagging and boosting models. Also report the mean and standard deviation of test accuracy across the 3 refits.

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import perf_counter
from sklearn.tree import DecisionTreeClassifier
from joblib import Parallel, delayed

train = np.load("../Data/X_train.npy.npz")
X_train = train["X_train"]
y_train = train["y_train"]

valid = np.load("../Data/X_valid.npy.npz")
X_valid = valid["X_valid"]
y_valid = valid["y_valid"]

test = np.load("../Data/X_test.npy.npz")
X_test = test["X_test"]
y_test = test["y_test"]

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_valid.shape, y_valid.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (240, 4096) (240,)
Validation: (80, 4096) (80,)
Test: (80, 4096) (80,)


In [33]:
from common.Bagging import Bagging

In [34]:
from common.Boosting import Boosting

In [35]:
CLASSES = np.arange(40)

def align_class_probabilities(model, X):
    """Return probability vectors with a fixed 40-class column order."""
    probabilities = model.predict_proba(X)
    aligned = np.zeros((len(X), len(CLASSES)))

    for j, class_label in enumerate(model.classes_):
        aligned[:, int(class_label)] = probabilities[:, j]

    return aligned

In [36]:
def tree_class_vectors(model, X):
    """Class probability vectors for the single decision tree."""
    return align_class_probabilities(model, X)


def bagging_class_vectors(trees, X):
    """Normalized class-vote vectors for bagging."""
    votes = get_bagging_votes(trees, X)
    vectors = np.zeros((len(X), 40))

    for i, row in enumerate(votes):
        counts = np.bincount(row.astype(int), minlength=40)
        vectors[i] = counts / len(trees)

    return vectors


def samme_class_vectors(model, X):
    """Normalized weighted class-vote vectors for SAMME."""
    trees = model["trees"]
    alphas = np.asarray(model["alphas"], dtype=float)

    if len(trees) == 0 or alphas.sum() <= 0:
        raise ValueError("SAMME model has no valid weak learners.")

    vectors = np.zeros((len(X), 40))
    sample_indices = np.arange(len(X))

    for tree, alpha in zip(trees, alphas):
        predictions = tree.predict(X).astype(int)
        vectors[sample_indices, predictions] += alpha

    return vectors / alphas.sum()

In [40]:
from common.Bagging import Bagging
from common.Boosting import Boosting

In [41]:
def get_bagging_votes(trees, X):
    """
    DESCR:
        A utility function to determine the class for each sample
        using each Decision Tree in the ensemble.

    ARGS:
        trees: Ensemble of Decision Trees
        X: Features of the dataset

    RETURNS:
        A matrix of shape (No. of trees, No. of samples in X)
    """
    return np.array([
        tree.predict(X) for tree in trees
    ]).T

In [50]:
def train_samme(X, y, max_depth, n_rounds, random_state=26001):
    model = Boosting(
        X, y, max_depth, n_rounds, random_state
    )
    return model.result

In [51]:
N_JOBS = 3
BAGGING_SIZE = 50
SAMME_DEPTH = 4
SAMME_ROUNDS = 100

X_test_fixed = X_test.copy()
y_test_fixed = y_test.copy()

def run_one_repetition(r, X_r, y_r):
    # Use independent, reproducible seeds for each repetition
    seed = 26001 + r

    # 1. Single decision tree
    tree_r = DecisionTreeClassifier(
        max_depth=None,
        min_samples_leaf=2,
        random_state=seed
    )
    tree_r.fit(X_r, y_r)
    tree_vectors = tree_class_vectors(tree_r, X_test_fixed)

    # 2. Bagging ensemble
    bagging_r = Bagging(
        X_r,
        y_r,
        n_estimators=BAGGING_SIZE,
        rng=np.random.default_rng(seed)
    )
    bagging_vectors = bagging_class_vectors(
        bagging_r["trees"], X_test_fixed
    )

    # 3. SAMME boosting
    boosting_r = train_samme(
        X_r,
        y_r,
        max_depth=SAMME_DEPTH,
        n_rounds=SAMME_ROUNDS,
        random_state=seed
    )
    boosting_vectors = samme_class_vectors(
        boosting_r, X_test_fixed
    )

    return r, tree_vectors, bagging_vectors, boosting_vectors


results = Parallel(
    n_jobs=N_JOBS,
    backend="loky",
    verbose=10
)(
    delayed(run_one_repetition)(r, X_r, y_r)
    for r, (X_r, y_r) in enumerate(resampled_train_sets)
)

# Restore repetition order and stack the outputs
results.sort(key=lambda item: item[0])

tree_vectors_all = np.stack([item[1] for item in results])
bagging_vectors_all = np.stack([item[2] for item in results])
boosting_vectors_all = np.stack([item[3] for item in results])

print("Tree:", tree_vectors_all.shape)
print("Bagging:", bagging_vectors_all.shape)
print("Boosting:", boosting_vectors_all.shape)

[Parallel(n_jobs=3)]: Using backend LokyBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done   2 tasks      | elapsed:  2.3min
[Parallel(n_jobs=3)]: Done   7 tasks      | elapsed:  7.1min
[Parallel(n_jobs=3)]: Done  12 tasks      | elapsed:  9.5min
[Parallel(n_jobs=3)]: Done  19 tasks      | elapsed: 16.7min
[Parallel(n_jobs=3)]: Done  30 out of  30 | elapsed: 24.3min finished


Tree: (30, 80, 40)
Bagging: (30, 80, 40)
Boosting: (30, 80, 40)


In [53]:
def measure_bias_variance(vectors_all, y_true):
    """
    vectors_all: (repetitions, test_samples, classes)
    y_true:      true identity for each test sample
    """
    n_repeats, n_samples, n_classes = vectors_all.shape

    mean_vectors = vectors_all.mean(axis=0)
    one_hot = np.eye(n_classes)[y_true]

    bias_squared = np.mean(
        np.sum((mean_vectors - one_hot) ** 2, axis=1)
    )

    variance = np.mean(
        np.sum(
            (vectors_all - mean_vectors[None, :, :]) ** 2,
            axis=2
        )
    )

    predictions = np.argmax(vectors_all, axis=2)
    accuracies = np.mean(predictions == y_true[None, :], axis=1)

    return {
        "Bias squared": bias_squared,
        "Variance": variance,
        "Mean test accuracy (%)": accuracies.mean() * 100,
        "Std test accuracy (percentage points)": (
            accuracies.std(ddof=1) * 100
        )
    }

In [54]:
q3a_results = pd.DataFrame({
    "Single tree": measure_bias_variance(
        tree_vectors_all, y_test_fixed
    ),
    "Bagging (B=50)": measure_bias_variance(
        bagging_vectors_all, y_test_fixed
    ),
    "SAMME (depth=4, rounds=100)": measure_bias_variance(
        boosting_vectors_all, y_test_fixed
    )
}).T

display(q3a_results.round(4))

,Bias squared,Variance,Mean test accuracy (%),Std test accuracy (percentage points)
Single tree,0.4026,0.6433,47.7083,5.3041
Bagging (B=50),0.5197,0.0315,72.7083,4.1792
"SAMME (depth=4, rounds=100)",0.8813,0.0176,32.5417,6.8030


Across the 30 resampled training sets, bagging achieved the highest mean test accuracy of 72.71%, compared with 47.71% for a single decision tree and 32.54% for SAMME. Bagging also reduced the variance from 0.6433 to 0.0315, a reduction of approximately 95%. The standard deviation of test accuracy decreased from 5.30 to 4.18 percentage points. This shows that bagging improved prediction accuracy and made the results more stable across different training sets.

SAMME had the lowest variance (0.0176), but it also had the lowest mean test accuracy (32.54%) and the highest bias squared (0.8813). This shows that low variance does not always mean better accuracy. A model may give consistent predictions but still make many incorrect predictions. Overall, bagging performed best among the three methods because it achieved the highest test accuracy with much lower variance than a single decision tree.

In [56]:
DEPTHS = [2, 8, None]
BAGGING_SIZE_Q3B = 100
N_JOBS_Q3B = 4

X_test_fixed = X_test.copy()
y_test_fixed = y_test.copy()
N_CLASSES = 40


def bagging_at_depth(X, y, n_estimators, max_depth, seed):
    """Train a bootstrap ensemble using the specified tree depth."""
    rng = np.random.default_rng(seed)
    n_samples = len(y)
    trees = []

    for _ in range(n_estimators):
        indices = rng.choice(n_samples, size=n_samples, replace=True)
        tree_seed = int(rng.integers(0, 2**31 - 1))

        model = DecisionTreeClassifier(
            max_depth=max_depth,
            min_samples_leaf=2,
            random_state=tree_seed
        )
        model.fit(X[indices], y[indices])
        trees.append(model)

    return trees


def tree_vote_vectors(trees, X):
    """Return normalized class-vote vectors for a bagged ensemble."""
    votes = np.array([tree.predict(X) for tree in trees]).T
    vectors = np.zeros((len(X), N_CLASSES))

    for i, row in enumerate(votes):
        vectors[i] = (
            np.bincount(row.astype(int), minlength=N_CLASSES)
            / len(trees)
        )

    return vectors


def leaf_occupancy(model, X):
    """Measure leaf occupancy on the supplied training data."""
    leaf_ids = model.apply(X)
    _, counts = np.unique(leaf_ids, return_counts=True)

    return {
        "leaf_count": len(counts),
        "mean_samples_per_leaf": float(counts.mean()),
        "singleton_leaf_fraction": float(np.mean(counts == 1))
    }


def pairwise_disagreement(trees, X):
    """Mean pairwise prediction disagreement on X."""
    predictions = np.array([tree.predict(X) for tree in trees])
    n_trees = predictions.shape[0]

    if n_trees < 2:
        return 0.0

    # For each test example, calculate the fraction of disagreeing pairs.
    total_pairs = n_trees * (n_trees - 1) / 2
    disagreements = []

    for row in predictions.T:
        _, counts = np.unique(row, return_counts=True)
        agreeing_pairs = np.sum(counts * (counts - 1) / 2)
        disagreements.append(
            (total_pairs - agreeing_pairs) / total_pairs
        )

    return float(np.mean(disagreements))


def run_depth_repetition(depth, r, X_r, y_r):
    """Run one repetition for one depth; safe to parallelize."""
    seed = 26001 + r

    # Single tree
    single = DecisionTreeClassifier(
        max_depth=depth,
        min_samples_leaf=2,
        random_state=seed
    )
    single.fit(X_r, y_r)

    single_vectors = tree_class_vectors(single, X_test_fixed)
    single_accuracy = np.mean(
        np.argmax(single_vectors, axis=1) == y_test_fixed
    )
    single_occ = leaf_occupancy(single, X_r)

    # Bagged ensemble
    trees = bagging_at_depth(
        X_r, y_r,
        n_estimators=BAGGING_SIZE_Q3B,
        max_depth=depth,
        seed=seed
    )

    bagged_vectors = tree_vote_vectors(trees, X_test_fixed)
    bagged_accuracy = np.mean(
        np.argmax(bagged_vectors, axis=1) == y_test_fixed
    )

    # Average occupancy across the 100 constituent trees
    occs = [leaf_occupancy(tree, X_r) for tree in trees]

    bagged_occ = {
        "leaf_count": np.mean([o["leaf_count"] for o in occs]),
        "mean_samples_per_leaf": np.mean(
            [o["mean_samples_per_leaf"] for o in occs]
        ),
        "singleton_leaf_fraction": np.mean(
            [o["singleton_leaf_fraction"] for o in occs]
        )
    }

    disagreement = pairwise_disagreement(trees, X_test_fixed)

    return {
        "depth": depth,
        "repetition": r,
        "single_vectors": single_vectors,
        "bagged_vectors": bagged_vectors,
        "single_accuracy": single_accuracy,
        "bagged_accuracy": bagged_accuracy,
        "single_occ": single_occ,
        "bagged_occ": bagged_occ,
        "disagreement": disagreement
    }


# Run the independent depth/repetition combinations concurrently.
tasks = [
    (depth, r, X_r, y_r)
    for depth in DEPTHS
    for r, (X_r, y_r) in enumerate(resampled_train_sets)
]

parallel_results = Parallel(
    n_jobs=N_JOBS_Q3B,
    backend="loky",
    verbose=10
)(
    delayed(run_depth_repetition)(depth, r, X_r, y_r)
    for depth, r, X_r, y_r in tasks
)

print(f"Completed {len(parallel_results)} depth/repetition jobs.")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:  2.9min
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:  3.6min
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:  4.3min
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:  4.7min
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:  7.0min
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed: 10.1min
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed: 16.3min
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed: 26.8min
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed: 47.0min
[Parallel(n_jobs=4)]: Done  90 out of  90 | elapsed: 63.8min finished


Completed 90 depth/repetition jobs.


In [57]:
depth_results = {}

for depth in DEPTHS:
    items = sorted(
        [x for x in parallel_results if x["depth"] == depth],
        key=lambda x: x["repetition"]
    )

    single_vectors = np.stack(
        [x["single_vectors"] for x in items]
    )
    bagged_vectors = np.stack(
        [x["bagged_vectors"] for x in items]
    )

    single_metrics = measure_bias_variance(
        single_vectors, y_test_fixed
    )
    bagged_metrics = measure_bias_variance(
        bagged_vectors, y_test_fixed
    )

    depth_results[depth] = {
        "single_vectors": single_vectors,
        "bagged_vectors": bagged_vectors,
        "single_metrics": single_metrics,
        "bagged_metrics": bagged_metrics,

        "single_accuracy_mean": np.mean(
            [x["single_accuracy"] for x in items]
        ) * 100,
        "single_accuracy_std": np.std(
            [x["single_accuracy"] for x in items], ddof=1
        ) * 100,

        "bagged_accuracy_mean": np.mean(
            [x["bagged_accuracy"] for x in items]
        ) * 100,
        "bagged_accuracy_std": np.std(
            [x["bagged_accuracy"] for x in items], ddof=1
        ) * 100,

        "single_leaf_count": np.mean([
            x["single_occ"]["leaf_count"] for x in items
        ]),
        "single_mean_occupancy": np.mean([
            x["single_occ"]["mean_samples_per_leaf"] for x in items
        ]),
        "single_singleton_fraction": np.mean([
            x["single_occ"]["singleton_leaf_fraction"] for x in items
        ]),

        "bagged_leaf_count": np.mean([
            x["bagged_occ"]["leaf_count"] for x in items
        ]),
        "bagged_mean_occupancy": np.mean([
            x["bagged_occ"]["mean_samples_per_leaf"] for x in items
        ]),
        "bagged_singleton_fraction": np.mean([
            x["bagged_occ"]["singleton_leaf_fraction"] for x in items
        ]),
        "bagged_disagreement": np.mean([
            x["disagreement"] for x in items
        ])
    }

print("Q3(b) results reconstructed successfully.")

Q3(b) results reconstructed successfully.


In [58]:
rows = []

for depth in DEPTHS:
    result = depth_results[depth]
    sm = result["single_metrics"]
    bm = result["bagged_metrics"]

    rows.append({
        "Depth": str(depth),
        "Model": "Single tree",
        "Bias squared": sm["Bias squared"],
        "Variance": sm["Variance"],
        "Mean accuracy (%)": result["single_accuracy_mean"],
        "Accuracy SD (pp)": result["single_accuracy_std"],
        "Mean leaves": result["single_leaf_count"],
        "Mean samples/leaf": result["single_mean_occupancy"],
        "Singleton leaf fraction": result["single_singleton_fraction"],
        "Pairwise disagreement": np.nan
    })

    rows.append({
        "Depth": str(depth),
        "Model": "Bagging (B=100)",
        "Bias squared": bm["Bias squared"],
        "Variance": bm["Variance"],
        "Mean accuracy (%)": result["bagged_accuracy_mean"],
        "Accuracy SD (pp)": result["bagged_accuracy_std"],
        "Mean leaves": result["bagged_leaf_count"],
        "Mean samples/leaf": result["bagged_mean_occupancy"],
        "Singleton leaf fraction": result["bagged_singleton_fraction"],
        "Pairwise disagreement": result["bagged_disagreement"]
    })

q3b_table = pd.DataFrame(rows)
display(q3b_table.round(4))


,Depth,Model,Bias squared,Variance,Mean accuracy (%),Accuracy SD (pp),Mean leaves,Mean samples/leaf,Singleton leaf fraction,Pairwise disagreement
0,2,Single tree,0.9102,0.0385,6.2083,1.3327,3.0000,80.0000,0.0000,NaN
1,2,Bagging (B=100),0.9300,0.0122,14.2500,5.6152,3.1553,76.8933,0.0018,0.9602
2,8,Single tree,0.7827,0.1194,15.4583,3.3715,9.2333,26.0606,0.0000,NaN
3,8,Bagging (B=100),0.8020,0.0182,45.9167,5.3128,10.8567,22.4703,0.0051,0.9266
4,None,Single tree,0.4026,0.6433,47.7083,5.3041,40.0000,6.0000,0.0000,NaN
5,None,Bagging (B=100),0.5194,0.0237,75.5417,3.0913,44.5043,5.4028,0.0120,0.8179


In [59]:
variance_rows = []

for depth in DEPTHS:
    result = depth_results[depth]

    single_var = result["single_metrics"]["Variance"]
    bagged_var = result["bagged_metrics"]["Variance"]

    reduction_pct = (
        100 * (single_var - bagged_var) / single_var
        if single_var > 0 else np.nan
    )

    variance_rows.append({
        "Depth": str(depth),
        "Single-tree variance": single_var,
        "Bagged variance": bagged_var,
        "Variance reduction (%)": reduction_pct,
        "Single-tree accuracy (%)": result["single_accuracy_mean"],
        "Bagged accuracy (%)": result["bagged_accuracy_mean"],
        "Accuracy improvement (pp)": (
            result["bagged_accuracy_mean"]
            - result["single_accuracy_mean"]
        )
    })

variance_comparison = pd.DataFrame(variance_rows)
display(variance_comparison.round(4))

,Depth,Single-tree variance,Bagged variance,Variance reduction (%),Single-tree accuracy (%),Bagged accuracy (%),Accuracy improvement (pp)
0,2,0.0385,0.0122,68.3675,6.2083,14.2500,8.0417
1,8,0.1194,0.0182,84.7288,15.4583,45.9167,30.4583
2,None,0.6433,0.0237,96.3210,47.7083,75.5417,27.8333


Q3(b): Effect of Tree Depth on Bias, Variance, and Bagging

As tree depth increases, the variance of a single decision tree increases, while its test accuracy also improves. Bagging reduces variance at every depth, making the predictions more stable. The variance reduction is approximately 68.37% for depth 2, 84.73% for depth 8, and 96.32% for unrestricted trees.

Bagging also improves test accuracy at all three depths. The largest accuracy improvement is 30.46 percentage points at depth 8, while unrestricted trees achieve the highest bagged accuracy of 75.54%. Overall, bagging is effective at reducing variance and improving accuracy, especially for deeper trees.

Q3(c): Does the Deepest Tree Necessarily Yield the Largest Improvement from Bagging?

The deepest tree does not always get the largest improvement in accuracy from bagging. In our results, the depth-8 tree shows the largest improvement of 30.46 percentage points, compared with 27.83 percentage points for unrestricted trees and 8.04 percentage points for depth 2.

The unrestricted tree has the highest variance (0.6433), which bagging reduces by approximately 96.32%. However, depth 8 achieves the largest accuracy improvement. This shows that reducing variance does not always lead to the largest improvement in accuracy. Deeper trees can learn more detailed patterns, but the final performance also depends on how well the trees make predictions.

Overall, depth 8 benefits the most from bagging in terms of accuracy improvement, while unrestricted trees achieve the highest final test accuracy of 75.54%. Therefore, the benefit of bagging depends on tree depth and predictive performance, not only on variance.

In [60]:
q3d_table = pd.DataFrame([
    {
        "Model": "Single decision tree",
        "Clean accuracy (%)": 56.25,
        "Masked accuracy (%)": 36.25,
        "Bias squared": 0.4026,
        "Variance": 0.6433
    },
    {
        "Model": "Bagging (B=50)",
        "Clean accuracy (%)": 76.25,
        "Masked accuracy (%)": 66.25,
        "Bias squared": 0.5197,
        "Variance": 0.0315
    },
    {
        "Model": "SAMME boosting (depth=4, rounds=100)",
        "Clean accuracy (%)": 56.25,
        "Masked accuracy (%)": 48.75,
        "Bias squared": 0.8813,
        "Variance": 0.0176
    }
])

display(q3d_table.round(4))

,Model,Clean accuracy (%),Masked accuracy (%),Bias squared,Variance
0,Single decision tree,56.25,36.25,0.4026,0.6433
1,Bagging (B=50),76.25,66.25,0.5197,0.0315
2,"SAMME boosting (depth=4, rounds=100)",56.25,48.75,0.8813,0.0176


### Q3(d): Final Comparison of Single Tree, Bagging, and Boosting

Bagging performed best overall, achieving the highest accuracy on both the clean test set (76.25%) and the masked test set (66.25%). It also reduced variance significantly compared with a single decision tree. The single tree was more affected by the missing region, with accuracy dropping from 56.25% to 36.25%, a decrease of 20 percentage points. SAMME boosting had the lowest variance (0.0176), but its accuracy was lower than bagging on both test sets. Overall, these results show that **low variance does not necessarily mean high accuracy**. Bagging provided the best balance of accuracy and stability among the three models.